# Topic 3: Parallel MMLU Evaluation with Ollama

Two MMLU topics evaluated with three timing trials:
- **Trial 1**: HuggingFace functions (no Ollama)
- **Trial 2**: Ollama programs, sequential
- **Trial 3**: Ollama programs, parallel

In [6]:
!pip install -q datasets requests tqdm
!pip install -q transformers torch accelerate  # For Trial 1

In [9]:
!apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
The following additional packages will be installed:
  libpci3 pci.ids
The following NEW packages will be installed:
  libpci3 pci.ids pciutils
0 upgraded, 3 newly installed, 0 to remove and 38 not upgraded.
Need to get 343 kB of archives.
After this operation, 1,581 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Fetched 343 kB in 0s (1,227 kB/s)  
Selecting previously unselected package pci.ids.
(Reading database ... 126696 files and directories currently installed.)
Preparing to unpack .../pci.ids_0.0~2022.01.22-1ubuntu0.1_all.deb ...
Unpacking pci.id

---
## Trial 1: Sequential HuggingFace Functions (No Ollama)

Uses `meta-llama/Llama-3.2-1B-Instruct` loaded via `transformers`.  
Evaluates both topics sequentially in-process.

In [10]:
CHOICES = ["A", "B", "C", "D"]
# No question limit — evaluate all entries in each topic's test set
# (astronomy: 152 questions, business_ethics: 100 questions)

def format_mmlu_prompt(question, choices):
    """Adapted from Running_an_LLM/llama_mmlu_eval.py:667"""
    prompt = f"{question}\n\n"
    for label, choice in zip(CHOICES, choices):
        prompt += f"{label}. {choice}\n"
    prompt += "\nAnswer:"
    return prompt

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from getpass import getpass

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"

print("Please enter your HuggingFace token:")
print("(Get your token from: https://huggingface.co/settings/tokens)")
hf_token = getpass("HF Token: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, token=hf_token, torch_dtype=torch.float16, device_map="auto"
)
model.eval()

Please enter your HuggingFace token:
(Get your token from: https://huggingface.co/settings/tokens)


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [12]:
from datasets import load_dataset
import gc

def get_hf_prediction(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=1,
            pad_token_id=tokenizer.eos_token_id, do_sample=False
        )
    text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()[:1].upper()
    del inputs, outputs
    return text if text in CHOICES else "A"

def evaluate_astronomy():
    """Evaluate Llama 3.2-1B on all MMLU astronomy test entries (152 questions)"""
    topic = "astronomy"
    dataset = load_dataset("cais/mmlu", topic, split="test")
    correct, total = 0, 0
    for item in dataset:  # All entries, no limit
        pred = get_hf_prediction(format_mmlu_prompt(item["question"], item["choices"]))
        correct += (CHOICES.index(pred) == item["answer"]) if pred in CHOICES else 0
        total += 1
    acc = correct / total * 100
    print(f"[astronomy] {correct}/{total} = {acc:.2f}%")
    gc.collect()
    return acc

def evaluate_business_ethics():
    """Evaluate Llama 3.2-1B on all MMLU business_ethics test entries (100 questions)"""
    topic = "business_ethics"
    dataset = load_dataset("cais/mmlu", topic, split="test")
    correct, total = 0, 0
    for item in dataset:  # All entries, no limit
        pred = get_hf_prediction(format_mmlu_prompt(item["question"], item["choices"]))
        correct += (CHOICES.index(pred) == item["answer"]) if pred in CHOICES else 0
        total += 1
    acc = correct / total * 100
    print(f"[business_ethics] {correct}/{total} = {acc:.2f}%")
    gc.collect()
    return acc

In [13]:
%%time
print("=== Trial 1: Individual HuggingFace Functions: Astronomy ===")
evaluate_astronomy()

=== Trial 1: Individual HuggingFace Functions: Astronomy ===


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

astronomy/test-00000-of-00001.parquet:   0%|          | 0.00/28.3k [00:00<?, ?B/s]

astronomy/validation-00000-of-00001.parq(…):   0%|          | 0.00/6.05k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/16 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[astronomy] 75/152 = 49.34%
CPU times: user 4.52 s, sys: 282 ms, total: 4.8 s
Wall time: 8.94 s


49.34210526315789

In [14]:
%%time
print("=== Trial 1: Individual HuggingFace Functions: Business Ethics ===")
evaluate_business_ethics()

=== Trial 1: Individual HuggingFace Functions: Business Ethics ===


business_ethics/test-00000-of-00001.parq(…):   0%|          | 0.00/21.6k [00:00<?, ?B/s]

business_ethics/validation-00000-of-0000(…):   0%|          | 0.00/5.09k [00:00<?, ?B/s]

business_ethics/dev-00000-of-00001.parqu(…):   0%|          | 0.00/4.96k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

[business_ethics] 45/100 = 45.00%
CPU times: user 2.72 s, sys: 15.4 ms, total: 2.74 s
Wall time: 3.93 s


45.0

---
## Ollama Setup (for Trials 2 & 3)

In [15]:
import subprocess, time
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
print("Ollama server started")

Ollama server started


In [16]:
!ollama pull llama3.2:1b

In [21]:
# Write Ollama-based astronomy evaluator to disk
program1_code = '''
import requests
from datasets import load_dataset

TOPIC = "astronomy"
MODEL = "llama3.2:1b"
OLLAMA_URL = "http://localhost:11434/api/chat"
CHOICES = ["A", "B", "C", "D"]
# Evaluate all test entries (no limit)

def format_mmlu_prompt(question, choices):
    prompt = f"{question}\\n\\n"
    for label, choice in zip(CHOICES, choices):
        prompt += f"{label}. {choice}\\n"
    prompt += "\\nAnswer:"
    return prompt

def get_ollama_prediction(prompt):
    resp = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"num_predict": 1, "temperature": 0}
    })
    text = resp.json()["message"]["content"].strip()[:1].upper()
    return text if text in CHOICES else "A"

def evaluate_astronomy():
    dataset = load_dataset("cais/mmlu", TOPIC, split="test")
    correct, total = 0, 0
    for item in dataset:  # All entries
        pred = get_ollama_prediction(format_mmlu_prompt(item["question"], item["choices"]))
        correct += (CHOICES.index(pred) == item["answer"]) if pred in CHOICES else 0
        total += 1
    acc = correct / total * 100
    print(f"[{TOPIC}] {correct}/{total} = {acc:.2f}%")
    return acc

if __name__ == "__main__":
    evaluate_astronomy()
'''
with open("/content/program1.py", "w") as f:
    f.write(program1_code)
print("program1.py (astronomy) written")

program1.py (astronomy) written


In [22]:
# Write Ollama-based business_ethics evaluator to disk
program2_code = '''
import requests
from datasets import load_dataset

TOPIC = "business_ethics"
MODEL = "llama3.2:1b"
OLLAMA_URL = "http://localhost:11434/api/chat"
CHOICES = ["A", "B", "C", "D"]
# Evaluate all test entries (no limit)

def format_mmlu_prompt(question, choices):
    prompt = f"{question}\\n\\n"
    for label, choice in zip(CHOICES, choices):
        prompt += f"{label}. {choice}\\n"
    prompt += "\\nAnswer:"
    return prompt

def get_ollama_prediction(prompt):
    resp = requests.post(OLLAMA_URL, json={
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
        "options": {"num_predict": 1, "temperature": 0}
    })
    text = resp.json()["message"]["content"].strip()[:1].upper()
    return text if text in CHOICES else "A"

def evaluate_business_ethics():
    dataset = load_dataset("cais/mmlu", TOPIC, split="test")
    correct, total = 0, 0
    for item in dataset:  # All entries
        pred = get_ollama_prediction(format_mmlu_prompt(item["question"], item["choices"]))
        correct += (CHOICES.index(pred) == item["answer"]) if pred in CHOICES else 0
        total += 1
    acc = correct / total * 100
    print(f"[{TOPIC}] {correct}/{total} = {acc:.2f}%")
    return acc

if __name__ == "__main__":
    evaluate_business_ethics()
'''
with open("/content/program2.py", "w") as f:
    f.write(program2_code)
print("program2.py (business_ethics) written")

program2.py (business_ethics) written


---
## Trial 2: Sequential Ollama Execution

Runs `program1.py` then `program2.py` one after the other via `subprocess.run`.  
Wall time ≈ time(astronomy) + time(business_ethics).

In [23]:
%%time
import subprocess
print("=== Trial 2: Sequential Ollama ===")
r1 = subprocess.run(["python", "/content/program1.py"], capture_output=True, text=True)
r2 = subprocess.run(["python", "/content/program2.py"], capture_output=True, text=True)
print(r1.stdout)
print(r2.stdout)

=== Trial 2: Sequential Ollama ===
[astronomy] 28/152 = 18.42%

[business_ethics] 30/100 = 30.00%

CPU times: user 29.8 ms, sys: 11.5 ms, total: 41.3 ms
Wall time: 53.4 s


---
## Trial 3: Parallel Ollama Execution

Launches both programs simultaneously via `subprocess.Popen`, then waits for both to finish.  
Wall time ≈ max(time(astronomy), time(business_ethics)) — ideally ~50% of Trial 2.

In [24]:
%%time
import subprocess
print("=== Trial 3: Parallel Ollama ===")
p1 = subprocess.Popen(["python", "/content/program1.py"],
                      stdout=subprocess.PIPE, stderr=subprocess.PIPE)
p2 = subprocess.Popen(["python", "/content/program2.py"],
                      stdout=subprocess.PIPE, stderr=subprocess.PIPE)
out1, _ = p1.communicate()
out2, _ = p2.communicate()
print(out1.decode())
print(out2.decode())

=== Trial 3: Parallel Ollama ===
[astronomy] 28/152 = 18.42%

[business_ethics] 30/100 = 30.00%

CPU times: user 17.5 ms, sys: 6.96 ms, total: 24.4 ms
Wall time: 32.1 s
